## Old

In [7]:
# !pip install -q git+https://github.com/boudinfl/pke.git
# !python -m spacy download en_core_web_sm

In [8]:
# import pke

# # initialize keyphrase extraction model, here TopicRank
# extractor = pke.unsupervised.TopicRank()

# # load the content of the document, here document is expected to be a simple
# # test string and preprocessing is carried out using spacy

# text = "Hi, my name is Abraham. Also Abraham is cool. Why did Shoar do this? What is Abraham's full name"

# extractor.load_document(input=text, language='en')

# # keyphrase candidate selection, in the case of TopicRank: sequences of nouns
# # and adjectives (i.e. `(Noun|Adj)*`)
# extractor.candidate_selection()

# # candidate weighting, in the case of TopicRank: using a random walk algorithm
# extractor.candidate_weighting()

# # N-best selection, keyphrases contains the 10 highest scored candidates as
# # (keyphrase, score) tuples
# keyphrases = extractor.get_n_best(n=10)

In [9]:
# print(keyphrases)

## New

In [10]:
from transformers import (
    TokenClassificationPipeline,
    AutoModelForTokenClassification,
    AutoTokenizer,
)
from transformers.pipelines import AggregationStrategy
import numpy as np

# Define keyphrase extraction pipeline
class KeyphraseExtractionPipeline(TokenClassificationPipeline):
    def __init__(self, model, *args, **kwargs):
        super().__init__(
            model=AutoModelForTokenClassification.from_pretrained(model),
            tokenizer=AutoTokenizer.from_pretrained(model),
            *args,
            **kwargs
        )

    def postprocess(self, all_outputs):
        results = super().postprocess(
            all_outputs=all_outputs,
            aggregation_strategy=AggregationStrategy.SIMPLE,
        )
        return np.unique([result.get("word").strip() for result in results])


In [11]:
# Load pipeline
model_name = "ml6team/keyphrase-extraction-kbir-inspec"
extractor = KeyphraseExtractionPipeline(model=model_name)


In [12]:
import pickle

def load_from_pickle(filename):
    with open(filename, "rb") as pickle_file:
        # Deserialize (unpickle) the data from the file
        loaded_combined_data = pickle.load(pickle_file)
    return loaded_combined_data

train_data = load_from_pickle("train_data.pkl")

In [13]:
import string

def remove_punctuation_from_edges(phrase):
    # Define the set of punctuation characters
    punctuation_chars = set(string.punctuation)

    # Remove leading punctuation
    left_stripped_phrase = phrase.lstrip(string.punctuation)

    # Remove trailing punctuation
    right_stripped_phrase = left_stripped_phrase.rstrip(string.punctuation)

    return right_stripped_phrase

In [14]:
# wrong, true = 0, 0
correct = 0
total_gold = 0
total_pred = 0

for i, doc in enumerate(train_data):
  # doc[0] is the text, doc[1] is list of annos
  keyphrases_normal = extractor(doc[0])
  keyphrases = [remove_punctuation_from_edges(item.lower()) for item in keyphrases_normal]

  set_of_preds = set(keyphrases)
  total_pred += len(set_of_preds)

  # print(type(keyphrases))
  set_of_golds = set()
  for anno in doc[1]:
    start = anno[2]
    end = anno[3]
    keyphrase = remove_punctuation_from_edges(doc[0][start: end].lower())

    set_of_golds.add(keyphrase)

  correctly_predicted = set_of_preds.intersection(set_of_golds)

  correct += len(correctly_predicted)
  total_gold += len(set_of_golds)

  if not (i % 10):
    print(i)


precision = correct / total_pred
recall = correct / total_gold
    # if keyphrase in set_of_keys:
    #   true += 1
    # else:
    #   # print(keyphrase, set_of_keys)
    #   wrong += 1




0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340


In [17]:
print(((precision*recall)*2)/(precision+recall))

0.3177216598064301


In [18]:
print(set_of_golds )
print(set_of_preds.intersection(set_of_golds))

{'demonstrate a specific type of defect', 'characterization of surface roughness', 'porosity structures in parts', 'hip', 'preliminary results', 'very low average porosity levels', 'visualization of defects', 'electron beam melted samples', 'characterize am parts', 'comparison of the part to its design model', 'average porosity', 'porosity distribution', 'porosity images', 'hot isostatic pressing', 'am parts', 'not follow the build direction', 'electron beam raster and overlap pattern', 'porosity structure changes', 'microct', 'build direction', 'porosity in am components', 'recent reports'}
{'electron beam melted samples', 'hot isostatic pressing', 'porosity distribution', 'microct'}
